In [1]:
import os
import json

import requests
import weaviate
import weaviate.classes as wvc
from weaviate.classes.query import MetadataQuery, Filter
from weaviate.embedded import EmbeddedOptions

from getpass import getpass

In [2]:
os.environ['OPENAI_API_KEY'] = getpass('Provide OPEN_API_KEY')

In [7]:
def jprint(value):
    print(json.dumps(value, indent=2, default=str))


def load_tiny_jeopardy():
    url = "https://raw.githubusercontent.com/weaviate-tutorials/quickstart/main/data/jeopardy_tiny.json"
    return requests.get(url, timeout=30).json()


def load_jeopardy_1k():
    url = "https://raw.githubusercontent.com/weaviate-tutorials/intro-workshop/main/data/jeopardy_1k.json"
    return requests.get(url, timeout=30).json()


def connect_local():
    if "OPENAI_API_KEY" not in os.environ:
        raise RuntimeError("Set OPENAI_API_KEY before running this notebook.")

    headers = {"X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"]}

    return weaviate.connect_to_local(headers=headers)


def recreate_question_collection(client):
    if client.collections.exists("Question"):
        client.collections.delete("Question")

    return client.collections.create(
        name="Question",
        vector_config=wvc.config.Configure.Vectors.text2vec_openai(
            model="text-embedding-3-small"
        ),
        generative_config=wvc.config.Configure.Generative.openai(
            model="gpt-4o-mini"
        ),
        properties=[
            wvc.config.Property(name="question", data_type=wvc.config.DataType.TEXT),
            wvc.config.Property(name="answer", data_type=wvc.config.DataType.TEXT),
            wvc.config.Property(name="category", data_type=wvc.config.DataType.TEXT),
        ],
    )


def import_questions(collection, data, *, include_round=False):
    objects = []
    for d in data:
        obj = {
            "question": d["Question"],
            "answer": d["Answer"],
            "category": d["Category"],
        }
        if include_round:
            obj["round"] = d.get("Round", "")
        objects.append(obj)

    result = collection.data.insert_many(objects)
    if result.errors:
        print("Import errors:")
        jprint(result.errors)
    return result

In [9]:
client = connect_local()
questions = recreate_question_collection(client)
import_questions(questions, load_tiny_jeopardy())

BatchObjectReturn(_all_responses=[UUID('e03a0949-376e-4062-98aa-bdf422f3571f'), UUID('fb3804cf-1ef3-4815-82e2-97c331e7ccee'), UUID('81e4a049-62eb-4802-880f-477d8e635686'), UUID('769d1288-24af-43d3-ac07-600aae7797f8'), UUID('ade43141-78a4-4918-a854-8266565b9863'), UUID('e060716d-1eb8-48b2-99dc-c7322f7e66f8'), UUID('37a62291-ced4-4644-b9d7-4ded4a3b383d'), UUID('2ce22f11-e4ec-40e7-8438-98f8993594ce'), UUID('b5a20dd6-29e6-4c9d-bc3a-9c55828a0515'), UUID('84e46809-e91b-4151-961a-541294a4aef1')], elapsed_seconds=3.0515387058258057, errors={}, uuids={0: UUID('e03a0949-376e-4062-98aa-bdf422f3571f'), 1: UUID('fb3804cf-1ef3-4815-82e2-97c331e7ccee'), 2: UUID('81e4a049-62eb-4802-880f-477d8e635686'), 3: UUID('769d1288-24af-43d3-ac07-600aae7797f8'), 4: UUID('ade43141-78a4-4918-a854-8266565b9863'), 5: UUID('e060716d-1eb8-48b2-99dc-c7322f7e66f8'), 6: UUID('37a62291-ced4-4644-b9d7-4ded4a3b383d'), 7: UUID('2ce22f11-e4ec-40e7-8438-98f8993594ce'), 8: UUID('b5a20dd6-29e6-4c9d-bc3a-9c55828a0515'), 9: UUID('8

In [5]:
response = questions.query.near_text(
    query="animals",
    limit=2,
    return_properties=["question", "answer", "category"],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print(obj.metadata.distance, obj.properties)

0.5862237811088562 {'question': "It's the only living mammal in the order Proboseidea", 'answer': 'Elephant', 'category': 'ANIMALS'}
0.6441107988357544 {'question': 'Weighing around a ton, the eland is the largest species of this animal in Africa', 'answer': 'Antelope', 'category': 'ANIMALS'}


In [10]:
prompt = 'Tell me a short story about this answer: {answer}'

response = questions.generate.near_text(
    query="animals",
    single_prompt=prompt,
    limit=2,

)

for obj in response.objects:
    print('answer: ', obj.properties['answer'])
    print('generated: ', obj.generated)

answer:  Elephant
generated:  In the heart of the golden savannah, there was a wise old elephant named Elysia. Elysia was known throughout the land for her gentle spirit and vast knowledge of the earth’s many secrets. Animals from far and wide sought her counsel, whether it was to find water in the dry season or to navigate the dangers of the night.

One sunny afternoon, a young lion named Leo approached her, frustration written all over his face. “Elysia, why must you always be the one who knows everything? I want to be the smartest in the jungle!” 

Elysia, with her kind eyes and calm demeanor, simply chuckled. “Young Leo, wisdom is not about knowing everything. It’s about understanding your place in the world and learning from those around you.”

Unsatisfied, Leo challenged her. “Then teach me how to be wise!” 

Smiling, Elysia agreed. “Very well. Follow me.”

They wandered through the savannah, and Elysia pointed out various plants, animals, and insects. She told Leo of the interco

In [11]:
response = questions.generate.near_text(
    query="animals",
    grouped_task="Which of these categories would a zoologist be most interested in ? Explain briefly.",
    limit=10,
    return_properties=["category"],
)

print(response.generated)

A zoologist would be most interested in the category "ANIMALS." This is because zoology is the scientific study of animals, including their behavior, physiology, classification, and distribution. The questions and answers in the "ANIMALS" category, such as those about elephants, antelopes, and the gavial, directly relate to the study of animal species, their characteristics, and their habitats, which are central themes in zoology. In contrast, the "SCIENCE" category contains topics that, while relevant to biology as a whole, do not specifically focus on animal studies.


In [12]:
client.close()